# Your first scraper
In this project, we will guide you step by step through the process of:

1. creating a self-contained development environment.
1. retrieving some information from an API (a website for computers)
2. leveraging it to scrape a website that does not provide an API
3. saving the output for later processing

Here we query an API for a list of countries and their past leaders. We then extract and sanitize their short bio from Wikipedia. Finally, we save the data to disk.

This task is often the first (coding) step of a datascience project and you will often come back to it in the future.

You will study topics such as *scraping*, *data structures*, *regular expressions*, *concurrency* and *file handling*. We will point out useful resources at the appropriate time. 

Let's dive in!

## 0. Creating a clean environment

Use the [`venv`](https://docs.python.org/3/library/venv.html) command to create a new environment called `wikipedia_scraper_env`.

Activate it and add it to you `.gitignore` file. 

You will find more info about virtual environments in the course content and on the web.

## 1. API Scraping

### 1a. A simple API query
You will start with the basics: how to do a simple request to an [API endpoint](../../2.python/2.python_advanced/05.Scraping/5.apis.ipynb).

You will use the [requests](https://requests.readthedocs.io/en/latest/) external library through the `import` keyword. NOTE: external libraries need to be installed first. Check the [request Quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/) section of the documentation to:

1. Use the `get()` method to connect to this endpoint: https://country-leaders.onrender.com/status
2. Check if the `status_code` is equal to 200, which means OK.
    * if OK, `print()` the `text`` of the response.
    * if not, `print()` the `status_code`. 

Here is an explanation of [HTTP status codes](https://en.wikipedia.org/wiki/List_of_HTTP_status_codes).


In [978]:
import requests

root_url = "https://country-leaders.onrender.com"
status_url = root_url + "/status"

req = requests.get(status_url)

if req.status_code == 200:
    print(req.text)
else:
    print("Error:", req.status_code)

"Alive"


### 1b. Dealing with JSON

[JSON](https://quickref.me/json) is the preferred format to deal with data over the web. You cannot avoid it so you would better get acquainted.

Connect to another endpoint called `/countries` but this time the API will return data in the JSON format. 


In [979]:
countries_url = root_url + "/countries"
req = requests.get(countries_url)
countries = req.json()
print(req.status_code, countries)

403 {'message': 'The cookie is missing'}


### 1c. Cookies anyone?

It looks like the access to this API is restricted...
Query the `/cookie` endpoint and extract the appropriate field to access your cookie.

You will need to use this cookie in each of the following API requests.

In [980]:
cookie_url = root_url + "/cookie"

req = requests.get(cookie_url)
cookies = req.cookies

print(cookies)

<RequestsCookieJar[<Cookie user_cookie=98342741-7821-48cb-b47d-6d0d3583a0d9 for country-leaders.onrender.com/>]>


Try to query the countries endpoint using the cookie, save the output and print it.

In [981]:
req = requests.get(countries_url, cookies=cookies)
countries = req.json()

print(req.status_code, countries)

200 ['fr', 'us', 'be', 'ma', 'ru']


Chances are the cookie has expired... Thanksfully, you got a nice error message. For now, simply execute the last 2 cells quickly so you get a result.

### 1d. Getting the actual data from the API

Query the `/leaders` endpoint.

In [982]:
leaders_url = root_url + "/leaders"

req = requests.get(leaders_url, cookies=cookies, params={"country": "fr"})
leaders = req.json()

print(req.status_code)
print(leaders)

200
[{'id': 'Q157', 'first_name': 'François', 'last_name': 'Hollande', 'birth_date': '1954-08-12', 'death_date': None, 'place_of_birth': 'Rouen', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande', 'start_mandate': '2012-05-15', 'end_mandate': '2017-05-14'}, {'id': 'Q329', 'first_name': 'Nicolas', 'last_name': 'Sarkozy', 'birth_date': '1955-01-28', 'death_date': None, 'place_of_birth': 'Paris', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Nicolas_Sarkozy', 'start_mandate': '2007-05-16', 'end_mandate': '2012-05-15'}, {'id': 'Q2038', 'first_name': 'François', 'last_name': 'Mitterrand', 'birth_date': '1916-10-26', 'death_date': '1996-01-08', 'place_of_birth': 'Jarnac', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand', 'start_mandate': '1981-05-21', 'end_mandate': '1995-05-17'}, {'id': 'Q2042', 'first_name': 'Charles', 'last_name': 'de Gaulle', 'birth_date': '1890-11-22', 'death_date': '1970-11-09', 'place_of_birth': 'Lille', 'wikipedia_url'

It looks like this endpoint requires additional information in order to return its result. Check the API [*documentation*](https://country-leaders.onrender.com/docs) in your web browser.

Change the query to accept *parameters*. You should know where to find help by now.

In [983]:
leaders_per_country = {c: requests.get(leaders_url, cookies=cookies, params={"country": c}).json() for c in countries}

### 1e. A sneak peak at the data (finally)

Look inside a few examples. Notice the dictionary keys available for each entry. You have your first example of *structured data*. This data was sanitized for your benefit, meaning it is readily exploitable without modification.

You will also notice there is a Wikipedia link for each entry. You will need to extract additional information there. This will be a case of *semi-structured* data.

The /countries endpoint returns a `list` of several country codes.

You need to loop through this list and query the /leaders endpoint for each one. Save each `json` result in a dictionary called `leaders_per_country`.

In [984]:
def get_leaders():
    root_url = "https://country-leaders.onrender.com"
    cookie_url = root_url + "/cookie"
    countries_url = root_url + "/countries"
    leaders_url = root_url + "/leaders"

    # get a fresh cookie
    cookies = requests.get(cookie_url).cookies

    # get the list of countries
    countries = requests.get(countries_url, cookies=cookies).json()

    # loop over countries and collect their leaders
    leaders_per_country = {}
    for country in countries:
        leaders_per_country[country] = requests.get(
            leaders_url, cookies=cookies, params={"country": country}
        ).json()

    return leaders_per_country

In [985]:
leaders_per_country = get_leaders()
print(leaders_per_country.keys())

leaders_per_country["fr"][0]

dict_keys(['fr', 'us', 'be', 'ma', 'ru'])


{'id': 'Q157',
 'first_name': 'François',
 'last_name': 'Hollande',
 'birth_date': '1954-08-12',
 'death_date': None,
 'place_of_birth': 'Rouen',
 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande',
 'start_mandate': '2012-05-15',
 'end_mandate': '2017-05-14'}

It is finally time to create a `get_leaders()` function for the above code. You will build on it later-on. This function takes no parameter. Inside it, you will need to:
1. define the urls
2. get the cookies
2. get the countries
3. loop over them and save their leaders in a dictionary
4. return the dictionary

In [986]:
# pick the first French leader's Wikipedia URL
wikipedia_url = leaders_per_country["fr"][0]["wikipedia_url"]
print(wikipedia_url)

req = requests.get(wikipedia_url)
print(req.status_code)
print(req.text[:500])   # just the first 500 chars so it's readable

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
403
Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119.



Test your function, save the result in the `leaders_per_country` dictionary and check its ouput.

In [987]:
headers = {"User-Agent": "WikipediaScraperBot/1.0 (learning project)"}

req = requests.get(wikipedia_url, headers=headers)
print(req.status_code)
print(req.text[:500])

200
<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientp


## 2. Extracting data from Wikipedia

Query one of the leaders' Wikipedia urls and display its `text` (not JSON).

In [988]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(req.text, "html.parser")

paragraphs = soup.find_all("p")

print("Number of paragraphs:", len(paragraphs))
print(paragraphs[0].get_text()[:200])  

Number of paragraphs: 154




Ouch! You get the raw HTML code of the webpage. If you try to deal with it without tools, you will be there all night. Instead, use the [beautiful soup 4](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) *external* library. You will find more info about it [here](../../2.python/2.python_advanced/05.Scraping/1.beautifulsoup_basic.ipynb) and [here](../../2.python/2.python_advanced/05.Scraping/2.beautifulsoup_advanced.ipynb)

Using the Quickstart section, start by importing the library and loading the output of your `get_text()` function.

Use the `prettify()` function and print it to take a look. You will start the actual parsing in the next step.

In [989]:
first_paragraph = ""

for paragraph in paragraphs:
    if paragraph.find("b"):          
        first_paragraph = paragraph.get_text()
        break                         
print(first_paragraph)

test_url = leaders_per_country["us"][0]["wikipedia_url"]
html = requests.get(test_url, headers=headers).text
paragraphs = BeautifulSoup(html, "html.parser").find_all("p")

François Hollande [fʁɑ̃swa ɔlɑ̃d][n 3] Écouterⓘ, né le 12 août 1954 à Rouen (Seine-Inférieure), est un haut fonctionnaire et homme d'État français. Il est président de la République française du 15 mai 2012 au 14 mai 2017.



That looks better but you need to extract the right part of the webpage: the text of the first paragraph.

It is a bit tricky because Wikipedia pages slightly differ in structure from one language to the next. We cannot simply get the text for the first HTML paragraph.

You will start by getting all the HTML paragraphs from the HTML source and saving them in the `paragraphs` variable.

Use the documentation or google the appropriate keywords.

In [990]:
def get_first_paragraph(wikipedia_url):
    print(wikipedia_url)   

    headers = {"User-Agent": "WikipediaScraperBot/1.0 (learning project)"}
    html = requests.get(wikipedia_url, headers=headers).text
    soup = BeautifulSoup(html, "html.parser")
    paragraphs = soup.find_all("p")

    first_paragraph = ""
    for paragraph in paragraphs:
        if paragraph.find("b"):
            first_paragraph = paragraph.get_text()
            break

    return first_paragraph

If you try different urls, you might find that the paragraph you want may be at a different index each time.

That is where you need to be clever and ask yourself what would be a reliable way to identify the right index ie. which string matches only the first paragraph whatever the language...

Spend a good 30 minutes on the problem and brainstorm with your fellow learners. If you come out empty handed, ask your coach.

1. Loop over the HTML paragraphs
2. When you have identified the correct one:
   * Store the [text](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#output) inside the `first_paragraph` variable
   * Exit the loop

At this stage, you can create a function to maintain consistency in your code. We will give you its *skeleton*, you will copy the code you wrote and make it work inside a function.

Don't forget to test your function.

In [991]:
test_url = leaders_per_country["fr"][0]["wikipedia_url"]
print(get_first_paragraph(test_url))

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
François Hollande [fʁɑ̃swa ɔlɑ̃d][n 3] Écouterⓘ, né le 12 août 1954 à Rouen (Seine-Inférieure), est un haut fonctionnaire et homme d'État français. Il est président de la République française du 15 mai 2012 au 14 mai 2017.



### 2a. Regular expressions to the rescue

Now that you have extracted the content of the first paragraph, the only thing that remains to finish your Wikipedia scraper is to sanitize the output.

Indeed some Wikipedia references, HTML code, phonetic pronunciation etc. may linger. You might find *regular expressions* handy to get rid of them and obtain pristine text. You will find some useful documentation about regular expressions [here](../../2.python/2.python_advanced/03.Regex/regex.ipynb)

Once you have one of your regex working online, try it in the cell below. 

Hints: 
* Check the `sub()` method documentation.
* Make sure to test urls in different languages. Some may look good but other do not.

In [992]:
import re

text = get_first_paragraph(leaders_per_country["fr"][0]["wikipedia_url"])
print("BEFORE:", text[:200])

clean = re.sub(r"\[[^\]]*\]", "", text)
print("AFTER :", clean[:200])

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
BEFORE: François Hollande [fʁɑ̃swa ɔlɑ̃d][n 3] Écouterⓘ, né le 12 août 1954 à Rouen (Seine-Inférieure), est un haut fonctionnaire et homme d'État français. Il est président de la République française du 15 ma
AFTER : François Hollande  Écouterⓘ, né le 12 août 1954 à Rouen (Seine-Inférieure), est un haut fonctionnaire et homme d'État français. Il est président de la République française du 15 mai 2012 au 14 mai 201


Overwrite the `get_first_paragraph()` function by applying your regex to the first paragraph before returning it.

In [993]:
def clean_text(text):
    text = re.sub(r"\[[^\]]*\]", "", text)   
    text = text.replace("\xa0", " ")          
    text = re.sub(r"\s+", " ", text)         
    return text.strip()                      

Come up with other regexes to capture other patterns and sanitize the outputs completely. Modify your `get_first_paragraph()` function accordingly.

In [994]:
def get_first_paragraph(wikipedia_url):
    print(wikipedia_url)

    headers = {"User-Agent": "WikipediaScraperBot/1.0 (learning project)"}
    html = requests.get(wikipedia_url, headers=headers).text
    soup = BeautifulSoup(html, "html.parser")
    paragraphs = soup.find_all("p")

    first_paragraph = ""
    for paragraph in paragraphs:
        if paragraph.find("b"):
            first_paragraph = clean_text(paragraph.get_text())  
            break

    return first_paragraph

print(get_first_paragraph(leaders_per_country["fr"][0]["wikipedia_url"]))

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
François Hollande Écouterⓘ, né le 12 août 1954 à Rouen (Seine-Inférieure), est un haut fonctionnaire et homme d'État français. Il est président de la République française du 15 mai 2012 au 14 mai 2017.


## 3. Putting it all together

Let's go back to your `get_leaders()` function and update it with an *inner* loop over each leader. You will query the url provided and extract the first paragraph using the `get_first_paragraph()` function you just finished. You will then update that `leader`'s dictionary and move on to the next one.

Notice, the rest of the code should not change since you modify the leader's data one by one.

In [ ]:
def get_leaders():
    root_url = "https://country-leaders.onrender.com"
    cookie_url = root_url + "/cookie"
    countries_url = root_url + "/countries"
    leaders_url = root_url + "/leaders"

    cookies = requests.get(cookie_url).cookies
    countries = requests.get(countries_url, cookies=cookies).json()

    leaders_per_country = {}
    for country in countries:
        leaders = requests.get(
            leaders_url, cookies=cookies, params={"country": country}
        ).json()

        for leader in leaders:
            leader["first_paragraph"] = get_first_paragraph(leader["wikipedia_url"])

        leaders_per_country[country] = leaders

    return leaders_per_country

(wikipedia_url)


In [1001]:
leaders_per_country = get_leaders()
leaders_per_country["fr"][0]

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napol%C3%A9on_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/Ren%C3%A9_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/%C3%89mile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincar%C3%A9
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d%27%C3%89tat)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.

TypeError: string indices must be integers, not 'str'

Does the function crash in the middle of the loop? Chances are the cookies have expired while looping over the leaders.

Modify your function with an *exception* or check if the `status_code` is a cookie error. In either case, get new cookies and query the api again.

If your code did not crash,

In [1017]:
import requests
import re
from bs4 import BeautifulSoup


def clean_text(text):
    text = re.sub(r"\[[^\]]*\]", "", text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def get_first_paragraph(wikipedia_url, session):
    print(wikipedia_url)
    headers = {"User-Agent": "WikipediaScraperBot/1.0 (learning project)"}
    html = session.get(wikipedia_url, headers=headers).text
    soup = BeautifulSoup(html, "html.parser")
    paragraphs = soup.find_all("p")

    first_paragraph = ""
    for paragraph in paragraphs:
        if paragraph.find("b"):
            first_paragraph = clean_text(paragraph.get_text())
            break
    return first_paragraph


def get_leaders():
    root_url = "https://country-leaders.onrender.com"
    cookie_url = root_url + "/cookie"
    countries_url = root_url + "/countries"
    leaders_url = root_url + "/leaders"

    session = requests.Session()
    cookies = requests.get(cookie_url).cookies
    countries = requests.get(countries_url, cookies=cookies).json()

    leaders_per_country = {}
    for country in countries:
        req = requests.get(leaders_url, cookies=cookies, params={"country": country})

        # self-heal: if the cookie expired, get a fresh one and try again
        if req.status_code != 200:
            cookies = requests.get(cookie_url).cookies
            req = requests.get(leaders_url, cookies=cookies, params={"country": country})

        leaders = req.json()
        for leader in leaders:
            leader["first_paragraph"] = get_first_paragraph(leader["wikipedia_url"], session)
        leaders_per_country[country] = leaders

    return leaders_per_country


Check the output of your function again.

Well done! It took a while however... Let's speed things up. The main *bottleneck* is the loop. We call on the Wikipedia website many times.

You will use the same *session* to call all the wikipedia pages. Check the *Advanced Usage* section of the Requests module's documentation.

Start by modifying the `get_first_paragraph()` function to accept a session parameter and adjust the `get()` method call.

Modify your `get_leaders()` function to make use of a single session for all the Wikipedia calls.
1. create a `Session` object outside of the loop over countries.
2. pass it to the `get_first_paragraph()` function as an argument.

In [1018]:
leaders_per_country = get_leaders()
leaders_per_country["fr"][0]["first_paragraph"]


https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napol%C3%A9on_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/Ren%C3%A9_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/%C3%89mile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincar%C3%A9
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d%27%C3%89tat)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.

"François Hollande Écouterⓘ, né le 12 août 1954 à Rouen (Seine-Inférieure), est un haut fonctionnaire et homme d'État français. Il est président de la République française du 15 mai 2012 au 14 mai 2017."

Test your new functions.



## 4. Saving your hard work

The final step is to save the ``leaders_per_country`` dictionary in the `leaders.json` file using the [json](https://docs.python.org/3/library/json.html) module. Check out the `with` statement.

In [1019]:
import json

with open("leaders.json", "w", encoding="utf-8") as f:
    json.dump(leaders_per_country, f, ensure_ascii=False, indent=4)

print("Saved!")


Saved!


Make sure the file can be read back. Write the code to read the file. And check the variables are the same.

In [1023]:
def save(leaders_per_country, filepath="leaders.json"):
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(leaders_per_country, f, ensure_ascii=False, indent=4)

# call it
save(leaders_per_country)
print("Saved via function!")


Saved via function!


Make a function `save(leaders_per_country)` to call this code easily.

In [1024]:
# 3 lines


In [1025]:
# Call the function (1 line)


## 5. Tidy things up in a stand-alone python script

Congratulations! You now have a working scraper! However, your code is scattered throughout this notebook along side the tutorials. Hardly production ready...

Copy and paste what you need in a separate `leaders_scraper.py` file.
Make sure it works by calling `python3 leaders_scraper.py`

## (Optional) To go further

If you want to practice scraping, you can read this section and tackle the exercises.

1. Restructure your code by using OOP (see ReadMe).
2. You have noticed the API returns very partial results for country leaders. Many are missing. Overwrite the `get_leaders()` function to get its list from Wikipedia and extract their *personal details* from the frame on the side.

Good luck!